# 03 — Game Predictions

Two game-level XGBoost models:
1. **Game Winner** — Binary classifier (home win probability)
2. **Point Spread** — Regressor (predicted margin)

Both trained on matchup features with temporal train/val/test split.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
import joblib

from nba_predict.config import (
    MODELS_DIR, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS,
)
from nba_predict.features.matchup_features import build_matchup_dataset, get_feature_columns
from nba_predict.evaluation.metrics import (
    classification_metrics, regression_metrics, calibration_metrics, feature_importance,
)
from nba_predict.evaluation.baselines import always_home_baseline

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded.')

Libraries loaded.


In [2]:
# Load data and models
print("Building matchup dataset...")
df = build_matchup_dataset()

# Load models to get their specific feature sets
gw_artifact = joblib.load(MODELS_DIR / 'game_winner.joblib')
ps_artifact = joblib.load(MODELS_DIR / 'point_spread.joblib')
gw_model = gw_artifact['model']
ps_model = ps_artifact['model']
gw_features = gw_artifact['feature_cols']
ps_features = ps_artifact['feature_cols']
calibrator = gw_artifact.get('calibrator')

test_df = df[df['season'].isin(TEST_SEASONS)]
train_df = df[df['season'].isin(TRAIN_SEASONS)]

print(f"Game Winner features: {len(gw_features)}")
print(f"Point Spread features: {len(ps_features)}")
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

# Target distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['home_win'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#F44336', '#4CAF50'])
axes[0].set_title('Home Win Distribution (Train)')
axes[0].set_xticklabels(['Away Win', 'Home Win'], rotation=0)

margin_col = 'home_margin' if 'home_margin' in train_df.columns else 'margin'
train_margin = pd.to_numeric(train_df[margin_col], errors='coerce').dropna()
axes[1].hist(train_margin, bins=50, color='steelblue', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_title('Margin Distribution (Train)')
axes[1].set_xlabel('Home Margin')
plt.tight_layout()
plt.show()

Building matchup dataset...


Game Winner features: 37
Point Spread features: 22
Train: 25,111 | Test: 2,472


/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41270/2916953658.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1. Game Winner (Classification)

In [3]:
# Use only features the model was trained on
avail_gw = [f for f in gw_features if f in test_df.columns]
X_test_gw = test_df[avail_gw].astype(float)
y_test = test_df['home_win'].astype(int)

raw_prob = gw_model.predict_proba(X_test_gw)[:, 1]
y_prob = calibrator.predict(raw_prob) if calibrator else raw_prob
y_pred = (y_prob > 0.5).astype(int)

metrics = classification_metrics(y_test.values, y_pred, y_prob)
cal = calibration_metrics(y_test.values, y_prob)

print(f"Accuracy:    {metrics['accuracy']:.4f}")
print(f"AUC-ROC:     {metrics['auc_roc']:.4f}")
print(f"Log Loss:    {metrics['log_loss']:.4f}")
print(f"Brier Score: {cal['brier_score']:.4f}")
print(f"ECE:         {cal['ece']:.4f}")

Accuracy:    0.6598
AUC-ROC:     0.7176
Log Loss:    0.6306
Brier Score: 0.2130
ECE:         0.0274


In [4]:
# ROC Curve, Calibration, Confusion Matrix
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
axes[1].plot(prob_pred, prob_true, 's-', color='steelblue', label='Model')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title(f'Calibration Plot (ECE={cal["ece"]:.3f})')
axes[1].legend()

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Away', 'Home']).plot(ax=axes[2], cmap='Blues')
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41270/644988547.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Baseline comparison
baselines = {
    'Always Home': always_home_baseline(y_test.values)['accuracy'],
    'XGBoost (ours)': metrics['accuracy'],
}

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ccc', '#2196F3']
pd.Series(baselines).plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Accuracy')
ax.set_title('Game Winner: Model vs Baselines')
ax.set_xlim(0.45, 0.7)
for i, (name, val) in enumerate(baselines.items()):
    ax.text(val + 0.003, i, f'{val:.1%}', va='center')
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41270/972061670.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Feature importance — top 20
fi = feature_importance(gw_model, avail_gw, top_n=20)

fig, ax = plt.subplots(figsize=(10, 8))
fi_plot = fi.iloc[::-1]
ax.barh(fi_plot['feature'], fi_plot['importance'], color='steelblue', alpha=0.8)
ax.set_xlabel('Importance (gain)')
ax.set_title('Game Winner \u2014 Top 20 Features')
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41270/1143253632.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Point Spread (Regression)

In [7]:
avail_ps = [f for f in ps_features if f in test_df.columns]
X_test_ps = test_df[avail_ps].astype(float)

margin_col = 'home_margin' if 'home_margin' in test_df.columns else 'margin'
y_test_margin = pd.to_numeric(test_df[margin_col], errors='coerce').astype(float)
ps_pred = ps_model.predict(X_test_ps)

ps_metrics = regression_metrics(y_test_margin.values, ps_pred)
print(f"MAE:  {ps_metrics['mae']:.2f} pts")
print(f"RMSE: {ps_metrics['rmse']:.2f} pts")

derived_acc = ((ps_pred > 0) == (y_test_margin.values > 0)).mean()
print(f"Derived winner accuracy: {derived_acc:.4f}")

MAE:  10.96 pts
RMSE: 14.10 pts
Derived winner accuracy: 0.6549


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_margin, ps_pred, alpha=0.15, s=10, color='steelblue')
lims = [-50, 50]
axes[0].plot(lims, lims, 'r--', alpha=0.5)
axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual Margin')
axes[0].set_ylabel('Predicted Margin')
axes[0].set_title(f'Point Spread: Predicted vs Actual (MAE={ps_metrics["mae"]:.1f})')

residuals = y_test_margin.values - ps_pred
axes[1].hist(residuals, bins=50, color='teal', alpha=0.7, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_title(f'Residual Distribution (mean={residuals.mean():.2f})')

plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41270/3980062461.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Error Analysis

In [9]:
# Accuracy by game number bucket
test_analysis = test_df.copy()
test_analysis['correct'] = (y_pred == y_test.values).astype(int)
test_analysis['game_bucket'] = pd.cut(
    pd.to_numeric(test_analysis['home_game_num'], errors='coerce'),
    bins=[0, 20, 40, 60, 82],
    labels=['1-20', '21-40', '41-60', '61-82']
)

acc_by_bucket = test_analysis.groupby('game_bucket', observed=True)['correct'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

acc_by_bucket.plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].axhline(y=metrics['accuracy'], color='red', linestyle='--',
                label=f'Overall: {metrics["accuracy"]:.1%}')
axes[0].set_title('Accuracy by Season Phase')
axes[0].set_ylabel('Accuracy')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend()

# B2B impact
if 'away_is_b2b' in test_analysis.columns:
    b2b_acc = test_analysis.groupby('away_is_b2b')['correct'].mean()
    labels = ['Not B2B', 'Away B2B']
    axes[1].bar(labels[:len(b2b_acc)], b2b_acc.values, color='teal', alpha=0.8)
    axes[1].set_title('Accuracy: Away Team Back-to-Back')
    axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41270/3615587641.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Summary

**Key takeaways:**
- Differential features dominate: `diff_win_pct_season` is the single most important feature
- Model accuracy improves as the season progresses (more data for rolling features)
- Away back-to-back games are easier to predict (fatigue signal)
- Calibration is strong \u2014 probabilities are trustworthy